# Notebook 4 — Inferência LLM (Fase 3)

Avalia **Llama 8B** e **DeepSeek V3** no test set de 200 amostras.

**Protocolo** (instruções Eduardo):
- 20 seeds por configuração
- Temperatura 0.1
- Zero-shot e 50-shot
- Targets: TOperacao e TAtracado

In [1]:
import gc
import json
import os
import re
import time
import warnings
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from scipy.stats import wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
load_dotenv('.env')

# ── Caminhos ──────────────────────────────────────────────────────────────────
INPUT_DIR   = Path('data_input')
PREDS_DIR   = Path('preds')
METRICS_DIR = Path('metrics') / 'LLM'

for d in [PREDS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Região e modelos ──────────────────────────────────────────────────────────
REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

MODELS = {
    'qwen3_32b':   'qwen.qwen3-32b-v1:0',
    'deepseek_v3': 'deepseek.v3.2',
}

# ── Parâmetros ────────────────────────────────────────────────────────────────
TEMPERATURE = 0.1
N_SEEDS     = 20
SEEDS = [42, 137, 256, 512, 1024, 2048, 3141, 4096, 5000, 6273,
         7777, 8192, 9001, 10240, 11111, 12345, 13579, 14641, 15360, 16384]

TARGET_COLS = ['TOperacao', 'TAtracado']
TARGET_DESC = {
    'TOperacao': 'tempo de operação (T3: período de operação de carga/descarga)',
    'TAtracado': 'tempo total atracado no berço (T2+T3+T4)',
}

# TODOS os 6 targets excluídos do prompt — nenhum pode ser feature
ALL_TARGETS = [
    'TEstadia', 'TEsperaAtracacao', 'TAtracado',
    'TEsperaInicioOp', 'TOperacao', 'TEsperaDesatracacao',
]
EXCLUDE_COLS = ALL_TARGETS + ['IDAtracacao']
N_SHOT = 5

print('Configuração OK')
print(f'Região  : {REGION}')
print(f'Modelos : {list(MODELS.keys())}')
print(f'N_SHOT  : {N_SHOT}')
print(f'Excluídos do prompt: {EXCLUDE_COLS}')


Configuração OK
Região  : us-east-1
Modelos : ['qwen3_32b', 'deepseek_v3']
N_SHOT  : 5
Excluídos do prompt: ['TEstadia', 'TEsperaAtracacao', 'TAtracado', 'TEsperaInicioOp', 'TOperacao', 'TEsperaDesatracacao', 'IDAtracacao']


## 1. Carregar dados

In [2]:
df_test  = pq.read_table(str(INPUT_DIR / 'test_strings.parquet')).to_pandas()
df_train = pq.read_table(str(INPUT_DIR / 'train_strings.parquet')).to_pandas()

print(f'Test  : {df_test.shape}')
print(f'Train : {df_train.shape}')

# Todas as colunas exceto targets e ID — textos, numéricas, categóricas, OHE+SUM
PROMPT_COLS = [c for c in df_test.columns if c not in EXCLUDE_COLS]

print(f'\nColunas no prompt: {len(PROMPT_COLS)}')
print(PROMPT_COLS)


Test  : (200, 77)
Train : (391460, 77)

Colunas no prompt: 74
['Porto Atracação', 'Complexo Portuário', 'Tipo da Autoridade Portuária', 'Tipo de Operação', 'Tipo de Navegação da Atracação', 'Nacionalidade do Armador', 'Município', 'UF', 'SGUF', 'Região Geográfica', 'Região Hidrográfica', 'Instalação Portuária em Rio', 'Natureza da Carga', 'Percurso Transporte Interiores', 'STNaturezaCarga', 'Carga Geral Acondicionamento', 'lon', 'lat', 'mes_sin', 'mes_cos', 'Ano', 'Mes_num', 'DiaSemana', 'VLPesoCargaBruta', 'TEU', 'QTCarga', 'VLPesoCargaConteinerizada_total', 'valor_mov_regiao_total', 'valor_mov_rio_total', 'valor_mov_hidrovia_total', 'valor_mov_total_hidrografia', 'ohe_Tipo_Operação_da_Carga__Abastecimento', 'ohe_Tipo_Operação_da_Carga__Apoio', 'ohe_Tipo_Operação_da_Carga__Baldeação_de_Carga_Estrangeira_de_Passagem', 'ohe_Tipo_Operação_da_Carga__Baldeação_de_Carga_Nacional', 'ohe_Tipo_Operação_da_Carga__Cabotagem', 'ohe_Tipo_Operação_da_Carga__Interior', 'ohe_Tipo_Operação_da_Carga__L

## 2. Funções de prompt e parser

In [3]:
def build_row_text(row):
    """Formata uma linha como JSON compacto — reduz tokens mantendo todas as colunas."""
    d = {}
    for col in PROMPT_COLS:
        val = row.get(col, 'N/A')
        if pd.isna(val) or str(val).strip() in ('', 'nan', 'None'):
            val = 'N/A'
        elif isinstance(val, (np.floating, np.integer)):
            val = val.item()  # converte numpy -> Python nativo
        d[col] = val
    return json.dumps(d, ensure_ascii=False, separators=(',', ':'))


def build_zero_shot_prompt(row, target, seed):
    desc = TARGET_DESC[target]
    return (
        f'seed={seed}\n'
        f'Você é um especialista em operações portuárias brasileiras.\n'
        f'Estime o {desc} de uma atracação em horas.\n\n'
        f'Dados da atracação (JSON):\n{build_row_text(row)}\n\n'
        f'Responda APENAS com um número decimal em horas (ex: 18.5). Sem texto adicional.'
    )


def build_few_shot_prompt(row, target, examples_df, seed):
    """Seleciona N_SHOT exemplos do treino (por seed) e constrói o prompt few-shot."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(examples_df), size=N_SHOT, replace=False)
    examples = examples_df.iloc[idx]

    desc = TARGET_DESC[target]
    lines = [
        f'seed={seed}',
        f'Você é um especialista em operações portuárias brasileiras.',
        f'Estime o {desc} de uma atracação em horas.',
        f'',
        f'Abaixo estão {N_SHOT} exemplos reais (JSON):',
        f'',
    ]
    for _, ex in examples.iterrows():
        lines.append(build_row_text(ex))
        lines.append(f'→ {target}: {ex[target]:.2f}h')
        lines.append('')

    lines += [
        'Agora estime para a seguinte atracação:',
        '',
        build_row_text(row),
        '',
        'Responda APENAS com um número decimal em horas (ex: 18.5). Sem texto adicional.',
    ]
    return '\n'.join(lines)


def parse_response(text):
    """Extrai o primeiro número decimal da resposta do LLM."""
    if text is None:
        return np.nan
    text = text.replace(',', '.')
    matches = re.findall(r'-?\d+\.?\d*', text)
    if not matches:
        return np.nan
    val = float(matches[0])
    if val < 0 or val > 8760:
        return np.nan
    return val


print('Funções de prompt OK')

Funções de prompt OK


## 3. Função de inferência Bedrock

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

runtime = boto3.client('bedrock-runtime', region_name=REGION)


def call_llm(model_id, prompt, max_retries=8):
    """Chama o modelo via converse API com retry exponencial em caso de throttling."""
    for attempt in range(max_retries):
        try:
            response = runtime.converse(
                modelId=model_id,
                messages=[{
                    'role': 'user',
                    'content': [{'text': prompt}],
                }],
                inferenceConfig={
                    'maxTokens': 32,
                    'temperature': TEMPERATURE,
                },
            )
            return response['output']['message']['content'][0]['text'].strip()
        except runtime.exceptions.ThrottlingException:
            wait = min(2 ** attempt, 60)
            print(f'  Throttling — aguardando {wait}s...')
            time.sleep(wait)
        except Exception as e:
            print(f'  Erro: {e}')
            return None
    return None


def _infer_row(args):
    """Worker: infere uma única row. Retorna (idx, val)."""
    i, row, model_id, strategy, target, train_df, seed = args
    if strategy == 'zero_shot':
        prompt = build_zero_shot_prompt(row, target, seed)
    else:
        prompt = build_few_shot_prompt(row, target, train_df, seed)

    raw = call_llm(model_id, prompt)
    val = parse_response(raw)
    if np.isnan(val):
        val = float(np.median(train_df[target]))
    return i, val


def run_inference(model_name, model_id, strategy, target, seeds, df_test, df_train,
                  n_workers=3):
    y_true   = df_test[target].to_numpy(dtype=np.float32)
    results  = []
    n_test   = len(df_test)
    csv_path = METRICS_DIR / f'results_{strategy}.csv'

    print(f'\n=== {model_name} | {strategy} | {target} ===')

    for seed in seeds:
        pred_file = PREDS_DIR / f'{model_name}_{strategy}_{target}_seed{seed}.npy'

        if pred_file.exists():
            preds = np.load(pred_file)
            print(f'  seed={seed} (carregado do disco)')
        else:
            preds = [None] * n_test
            print(f'  seed={seed} — {n_test} amostras, {n_workers} threads...')

            rows = [(i, row, model_id, strategy, target, df_train, seed)
                    for i, (_, row) in enumerate(df_test.iterrows())]

            done       = 0
            batch_start = time.time()
            seed_start  = time.time()

            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = {ex.submit(_infer_row, r): r[0] for r in rows}
                for fut in as_completed(futures):
                    i, val = fut.result()
                    preds[i] = val
                    done += 1
                    if done % 20 == 0 or done == n_test:
                        elapsed = time.time() - batch_start
                        print(f'    [{done}/{n_test}]  batch={elapsed:.1f}s')
                        batch_start = time.time()

            total = time.time() - seed_start
            print(f'  seed={seed} concluído em {total:.1f}s')

            preds = np.array(preds, dtype=np.float32)
            np.save(pred_file, preds)

        rmse = float(np.sqrt(mean_squared_error(y_true, preds)))
        r2   = float(r2_score(y_true, preds))
        mae  = float(mean_absolute_error(y_true, preds))

        row_result = {
            'model': model_name, 'strategy': strategy,
            'target': target, 'seed': seed,
            'rmse': rmse, 'r2': r2, 'mae': mae,
        }
        results.append(row_result)
        print(f'  seed={seed}  RMSE={rmse:.3f}  R²={r2:.4f}')

        # Salva CSV incrementalmente após cada seed
        df_partial = pd.DataFrame(results)
        if csv_path.exists():
            df_existing = pd.read_csv(csv_path)
            df_combined = pd.concat([df_existing, df_partial], ignore_index=True)
            df_combined = df_combined.drop_duplicates(
                subset=['model', 'strategy', 'target', 'seed'], keep='last'
            )
            df_combined.to_csv(csv_path, index=False)
        else:
            df_partial.to_csv(csv_path, index=False)

    return results


print('Função de inferência OK')


# ── TESTE RÁPIDO ──────────────────────────────────────────────────────────────
for model_name, model_id in MODELS.items():
    print(f'[{model_name}]')
    raw = call_llm(model_id, 'Quanto é 1 + 1?')
    print(f'  Resposta: {raw!r}\n')

Função de inferência OK
[qwen3_32b]
  Resposta: '1 + 1 é igual a **2**.'

[deepseek_v3]
  Resposta: '1 + 1 = 2. 😊'



In [5]:
# ── TESTE RÁPIDO — verifica se os modelos respondem ──────────────────────────
for model_name, model_id in MODELS.items():
    print(f'[{model_name}]')
    raw = call_llm(model_id, 'Quanto é 1 + 1?')
    print(f'  Resposta: {raw!r}\n')

[qwen3_32b]
  Resposta: '1 + 1 é igual a **2**.'

[deepseek_v3]
  Resposta: '1 + 1 = 2. 😊'



## 4. Zero-shot

In [ ]:
results_zero = []

for model_name, model_id in MODELS.items():
    for target in TARGET_COLS:
        rows = run_inference(
            model_name, model_id,
            strategy='zero_shot',
            target=target,
            seeds=SEEDS,
            df_test     = df_test,
            df_train    = df_train,
            n_workers   = 1,
        )
        results_zero.extend(rows)

df_zero = pd.DataFrame(results_zero)
df_zero.to_csv(METRICS_DIR / 'results_zero_shot.csv', index=False)
print('\nZero-shot salvo.')
df_zero.groupby(['model','target'])[['rmse','r2','mae']].agg(['mean','std']).round(4)


=== qwen3_32b | zero_shot | TOperacao ===
  seed=42 (carregado do disco)
  seed=42  RMSE=36.214  R²=0.0105
  seed=137 (carregado do disco)
  seed=137  RMSE=36.105  R²=0.0164
  seed=256 — 200 amostras, 1 threads...
    [20/200]  batch=9.0s
    [40/200]  batch=9.0s
    [60/200]  batch=10.9s
    [80/200]  batch=8.7s
    [100/200]  batch=9.4s
    [120/200]  batch=9.9s
    [140/200]  batch=8.7s
    [160/200]  batch=13.6s
    [180/200]  batch=9.6s
    [200/200]  batch=9.7s
  seed=256 concluído em 98.3s
  seed=256  RMSE=36.714  R²=-0.0170
  seed=512 — 200 amostras, 1 threads...
    [20/200]  batch=10.8s
    [40/200]  batch=11.0s
    [60/200]  batch=10.7s
    [80/200]  batch=10.8s
    [100/200]  batch=9.8s
    [120/200]  batch=12.6s
    [140/200]  batch=10.4s
    [160/200]  batch=10.2s
    [180/200]  batch=10.3s
    [200/200]  batch=13.1s
  seed=512 concluído em 109.5s
  seed=512  RMSE=36.832  R²=-0.0236
  seed=1024 — 200 amostras, 1 threads...
    [20/200]  batch=10.8s
    [40/200]  batch=10

## 5. 5-shot

In [ ]:
results_few = []

for model_name, model_id in MODELS.items():
    for target in TARGET_COLS:
        rows = run_inference(
            model_name, model_id,
            strategy='few_shot',
            target=target,
            seeds=SEEDS,
            df_test=df_test,
            df_train=df_train,
            n_workers=1,
        )
        results_few.extend(rows)

df_few = pd.DataFrame(results_few)
df_few.to_csv(METRICS_DIR / 'results_few_shot.csv', index=False)
print('\n20-shot salvo.')
df_few.groupby(['model','target'])[['rmse','r2','mae']].agg(['mean','std']).round(4)

## 6. Resultados consolidados

In [ ]:
df_llm = pd.concat([df_zero, df_few], ignore_index=True)
df_llm.to_csv(METRICS_DIR / 'results_all_llm.csv', index=False)

print('=== RMSE médio por modelo, estratégia e target ===')
pivot = df_llm.groupby(['model','strategy','target'])['rmse'].mean().unstack('target').round(3)
print(pivot.to_string())

## 7. Wilcoxon LLM vs LLM

In [ ]:
def cohens_d(a, b):
    diff = a - b
    return float(diff.mean() / (diff.std(ddof=1) + 1e-12))


configs = df_llm[['model','strategy']].drop_duplicates().values.tolist()
rmse_by = {
    f'{m}_{s}': {
        t: df_llm[(df_llm.model==m) & (df_llm.strategy==s) & (df_llm.target==t)]
               .sort_values('seed')['rmse'].values
        for t in TARGET_COLS
    }
    for m, s in configs
}

from itertools import combinations
print('=== Wilcoxon pareado seed-a-seed (LLM vs LLM) ===')
for ka, kb in combinations(rmse_by.keys(), 2):
    print(f'\n{ka} vs {kb}:')
    for t in TARGET_COLS:
        a = rmse_by[ka][t]
        b = rmse_by[kb][t]
        if len(a) != len(b) or len(a) < 2:
            continue
        stat, p = wilcoxon(a, b)
        d = cohens_d(a, b)
        winner = ka if a.mean() < b.mean() else kb
        sig = '*' if p < 0.05 else 'n.s.'
        print(f'  {t:12s}  {ka}={a.mean():.3f}  {kb}={b.mean():.3f}  '
              f'p={p:.4f} {sig}  d={d:+.3f}  melhor={winner}')

print(f'\nMétricas em : {METRICS_DIR.resolve()}')
print(f'Predições em: {PREDS_DIR.resolve()}')